In [ ]:
import os

# copied from basic_ec.ipynb, but modified to try similar things on a replicated pool
# recreating https://bugzilla.redhat.com/show_bug.cgi?id=2277111
# created 16.7.25

# ---------- init a cluster with some data
#  (so that we can use manual commands)

# List the current directory
#print(os.listdir(.))

In [ ]:
# Ask the user to select a digit in the range 0 to 8
k = int(input("Please select a digit in the range 0 to 8: "))

# Check if the digit is within the valid range
if 0 <= k <= 8:
    print(f"will be running in k{k}")
else:
    print("Invalid selection. Please select a digit between 0 and 8.")
    exit(1)

from pathlib import Path

wd = Path(f"/home/rfriedma/src/k{k}/ceph/build")
os.chdir(wd)
%env CEPH_JTEST_ROOT=/home/rfriedma/src/k{k}/ceph/build
!echo $CEPH_JTEST_ROOT > /tmp/jpath


In [ ]:
!ls -l
!bash -c MGR=0 ls -l


In [ ]:
%%bash

function file_with_random_data() {
  local size_bytes=${1:-512}
  local file=$(mktemp)
  dd if=/dev/urandom of=$file bs=$size_bytes count=1
  printf '%s' "$file"
}


scrtch=to_"`date +%d_%H%M`"
echo $scrtch

MDS=0 MGR=1 OSD=3 MON=1 ../src/vstart.sh -n  --without-dashboard --msgr2 -X -o "osd_op_queue=wpq"
#MDS=0 MGR=1 OSD=3 MON=1 ../src/vstart.sh -n  --without-dashboard --msgr2 -X --memstore -o "memstore_device_bytes=68435456" -o "osd_op_queue=wpq"
sleep 2
bin/ceph -s

#bin/ceph tell osd.* config set debug_osd 20/20

bin/ceph config set global osd_pool_default_pg_autoscale_mode off
sleep 2

# disable rescheduling of the queue due to 'no-scrub' flags
#bin/ceph tell osd.* config set osd_scrub_backoff_ratio 0.9999

# initial set of global scrub scheduling parameters
bin/ceph tell osd.* config set osd_scrub_interval_randomize_ratio 0.1
bin/ceph tell osd.* config set osd_deep_scrub_randomize_ratio 0
bin/ceph tell osd.* config set osd_scrub_min_interval 1000
bin/ceph tell osd.* config set osd_scrub_max_interval 2000
bin/ceph tell osd.* config set osd_deep_scrub_interval 6000



In [ ]:
%%bash

function get_num_active_clean1() {
    local expression
    expression+="select(contains(\"active\") and contains(\"clean\")) | "
    expression+="select(contains(\"stale\") | not)"
    bin/ceph --format json pg dump pgs 2>/dev/null | \
        jq ".pg_stats | [.[] | .state | $expression] | length"
}

# create a replicated pool
bin/ceph osd pool create nonecpool 1 1 --autoscale_mode=off
bin/ceph osd pool set nonecpool pg_num 1
bin/ceph osd pool set nonecpool pg_autoscale_mode off
bin/ceph osd pool application enable nonecpool rados
sleep 1
echo "nonecpool ID: $(bin/ceph osd lspools | grep nonecpool | cut -d' '  -f 1)"
bin/ceph osd pool set nonecpool pg_num 1
bin/ceph osd pool set nonecpool pg_autoscale_mode off
# RRR must wait for clean here
sleep 2
echo "# active+clean: $(get_num_active_clean1)"
sleep 2
echo "# active+clean: $(get_num_active_clean1)"
bin/ceph tell osd.* config set debug_osd 10/10



In [ ]:
%%bash
function file_with_random_data() {
  local size_bytes=${1:-512}
  local file=$(mktemp)
  dd if=/dev/urandom of=$file bs=$size_bytes count=1
  printf '%s' "$file"
}

function get_num_active_clean() {
    local expression
    expression+="select(contains(\"active\") and contains(\"clean\")) | "
    expression+="select(contains(\"stale\") | not)"
    bin/ceph --format json pg dump pgs 2>/dev/null | \
        jq ".pg_stats | [.[] | .state | $expression] | length"
}


# create a test object
file=$(file_with_random_data 4000)
bin/rados -p nonecpool put dummy $file
sleep 1

#  find pg_id and primary OSD values by using the osd map command
bin/ceph osd map nonecpool dummy
bin/ceph --format=json-pretty osd map nonecpool dummy
#bin/ceph --format=json-pretty osd map nonecpool dummy | jq -r '.acting | .[0]'
targetp=$(bin/ceph --format=json-pretty osd map nonecpool dummy | jq -r '.acting_primary')
pgid=$(bin/ceph --format=json-pretty osd map nonecpool dummy | jq -r '.pgid')
echo "PG: $pgid - Primary OSD: $targetp"
echo "# active+clean: $(get_num_active_clean)"

bin/ceph tell osd.* config set debug_osd 20/20

bin/ceph pg $pgid query
bin/ceph tell $pgid deep-scrub
sleep 2
echo "# active+clean: $(get_num_active_clean)"



In [ ]:
%%bash
bin/rados -p nonecpool setxattr dummy x1 "vx1"
bin/rados -p nonecpool setxattr dummy x2 "vx2"
bin/rados -p nonecpool setxattr dummy x3 "vx3"
bin/rados -p nonecpool listxattr dummy



In [ ]:
%%bash
# stopping an OSD
function stop_osd() {
  local targetp=$1
  procid=$(cat out/osd.$targetp.pid)
  echo "Stopping OSD $targetp with PID $procid"
  kill -TERM $procid
}

prim_osd=$(bin/ceph --format=json-pretty osd map nonecpool dummy | jq -r '.acting_primary')
targetp=$prim_osd
#targetp=0 # we can try corrupting either the Primary or one of the Secondaries
echo "Stopping OSD $targetp"
stop_osd $targetp
sleep 1

bin/ceph-objectstore-tool --data-path dev/osd$targetp --pool nonecpool --op=info dummy
bin/ceph-objectstore-tool --data-path dev/osd$targetp --pgid 2.0_head --op info
echo "-"
echo "---------------------------------------------------"
bin/ceph-objectstore-tool --data-path dev/osd$targetp --pgid 2.0_head --op info dummy
bin/ceph-objectstore-tool --data-path dev/osd$targetp --op info dummy
bin/ceph-objectstore-tool --data-path dev/osd$targetp --pgid 2.0 '{"oid":"dummy","pool":2}' --op info
#bin/ceph-objectstore-tool --data-path dev/osd$targetp --pgid 2.0 '{"oid":"dummy","pool":2}' listxattr


In [ ]:
%%bash

# we can try corrupting either the Primary or one of the Secondaries
targetp=2
bin/ceph-objectstore-tool --data-path dev/osd$targetp --op list dummy
OH=$(bin/ceph-objectstore-tool --data-path dev/osd$targetp --op list dummy | sed -n 's/^[^{]*\({[^}]*}\).*$/\1/p')
echo "Object json name: $OH"

bin/ceph-objectstore-tool --data-path dev/osd$targetp --pgid 2.0 $OH list-attrs
bin/ceph-objectstore-tool --data-path dev/osd$targetp --pgid 2.0 $OH rm-attr "_x1"
bin/ceph-objectstore-tool --data-path dev/osd$targetp --pgid 2.0 $OH get-attr "_x2"
echo
bin/ceph-objectstore-tool --data-path dev/osd$targetp --pgid 2.0 $OH get-attr "_x3"
echo
bin/ceph-osd -i $targetp -c ./ceph.conf
sleep 3
bin/rados -p nonecpool listxattr dummy
bin/ceph tell osd.* config set debug_osd 20/20


#bin/ceph-objectstore-tool --data-path dev/osd2 --pgid 2.0s0 '{"oid":"dummy","pool":2,"shard_id":0}' get-attr "hinfo_key"
#bin/ceph-objectstore-tool --data-path dev/osd2 --pgid 2.0s0 '{"oid":"dummy","pool":2,"shard_id":0}' rm-attr "hinfo_key"
#sudo bin/ceph-objectstore-tool --data-path dev/osd2 --pgid 2.0s0 '{"oid":"dummy","pool":2}' rm-attr "hinfo_key"


In [ ]:
%%bash
lastp=2
bin/ceph-osd -i $lastp -c ./ceph.conf
sleep 3
bin/rados -p nonecpool listxattr dummy



In [ ]:
%%bash

lastp=2
bin/ceph-objectstore-tool --data-path dev/osd$lastp --op list dummy
OH=$(bin/ceph-objectstore-tool --data-path dev/osd$lastp --op list dummy | sed -n 's/^[^{]*\({[^}]*}\).*$/\1/p')
echo "Object json name: $OH"

bin/ceph-objectstore-tool --data-path dev/osd$lastp --pgid 2.0 $OH list-attrs
#bin/ceph-objectstore-tool --data-path dev/osd$lastp --pgid 2.0 $OH rm-attr "_x1"
bin/ceph-objectstore-tool --data-path dev/osd$lastp --pgid 2.0 $OH get-attr "_x2"



In [ ]:
%%bash

bin/ceph tell osd.* config set osd_scrub_auto_repair 0
bin/rados -p nonecpool listxattr dummy

bin/ceph pg ls-by-osd 1
bin/ceph tell 2.0 schedule-deep-scrub
sleep 4
bin/rados -p nonecpool listxattr dummy
echo "list-inconsistent"
bin/rados -p nonecpool list-inconsistent-obj 2.0 | jq '.'
bin/ceph pg ls-by-osd 1
bin/rados -p nonecpool getxattr dummy "x1"
bin/rados -p nonecpool getxattr dummy "x2"
bin/rados -p nonecpool getxattr dummy "x3"


In [ ]:
%%bash


#in Squid - this was enough to clear the inconsistency

bin/ceph tell osd.* config set osd_scrub_auto_repair 1
bin/rados -p nonecpool listxattr dummy
bin/ceph pg ls-by-osd 1
bin/ceph tell 2.0 deep-scrub
sleep 4
bin/rados -p nonecpool listxattr dummy
echo "list-inconsistent"
bin/rados -p nonecpool list-inconsistent-obj 2.0 | jq '.'
bin/ceph pg ls-by-osd 1
bin/rados -p nonecpool getxattr dummy "x1"
bin/rados -p nonecpool getxattr dummy "x2"
bin/rados -p nonecpool getxattr dummy "x3"
bin/ceph tell osd.* config set osd_scrub_auto_repair 1
sleep 3



In [ ]:
%%bash


bin/ceph tell osd.* config set osd_scrub_auto_repair 1
bin/rados -p nonecpool listxattr dummy
bin/ceph pg ls-by-osd 1
bin/ceph tell 2.0 schedule-deep-scrub
sleep 4
bin/rados -p nonecpool listxattr dummy
echo "list-inconsistent"
bin/rados -p nonecpool list-inconsistent-obj 2.0 | jq '.'
bin/ceph pg ls-by-osd 1
bin/rados -p nonecpool getxattr dummy "x1"
bin/rados -p nonecpool getxattr dummy "x2"
bin/rados -p nonecpool getxattr dummy "x3"
bin/ceph tell osd.* config set osd_scrub_auto_repair 1
sleep 3

echo "C --- cleanup"

bin/ceph pg ls-by-osd 1
bin/ceph tell 2.0 deep-scrub
sleep 4
bin/ceph pg ls-by-osd 1
bin/rados -p nonecpool list-inconsistent-obj 2.0 | jq '.'




In [ ]:
%%bash



echo "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX end of replicated_attrs_manip.ipynb"

In [ ]:
%%%bash
#PL1 is of size 3

bin/ceph osd pool create pl1 8 8
#bin/ceph osd pool autoscale-status
sleep 1
pl1_num=`bin/ceph osd pool stats pl1 | sed -n -r -e 's/.*id[^0-9]*([0-9]+)$/\1/p'`
echo $pl1_num
bin/ceph osd pool set pl1 size 3
bin/ceph osd pool set pl1 min_size 3
bin/ceph osd pool set pl1 pg_autoscale_mode off
bin/ceph osd pool stats
bin/ceph osd pool set pl1 noscrub 0
bin/ceph osd pool set pl1 nodeep-scrub 0
sleep 2

bin/rados bench -p pl1 -t 1 1 write -b 4096 --max-objects 8  --no-cleanup; 
bin/rados bench -p pl1 1 write -b 4096 --max-objects 128 --show-time --no-cleanup --run-name eeeee

#PL2

bin/ceph osd pool create pl2 8 8
bin/ceph osd pool set pl2 size 3
bin/ceph osd pool set pl2 min_size 3
bin/ceph osd pool set pl2 pg_autoscale_mode off
bin/ceph osd pool stats
bin/ceph osd pool set pl2 noscrub 0
bin/ceph osd pool set pl2 nodeep-scrub 0
sleep 2

bin/rados bench -p pl2 -t 1 1 write -b 4096 --max-objects 8  --no-cleanup; 
bin/rados bench -p pl2 1 write -b 4096 --max-objects 128 --show-time --no-cleanup --run-name eeeee
sleep 1

bin/ceph tell osd.* config set debug_osd 20/20
bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_00.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_01.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_02.json


In [ ]:
%%bash
echo '-------->' $CEPH_JTEST_ROOT
jp=$CEPH_JTEST_ROOT
if [ -z $CEPH_JTEST_ROOT ]; then
    echo "CEPH_JTEST_ROOT is not set"
    [[ -f /tmp/jpath ]] && jp=`cat /tmp/jpath` || jp='.'
    echo '-------->' $jp
    %env CEPH_JTEST_ROOT=$jp
fi
cd $jp


In [ ]:
%%bash

file_path="/tmp/jpath"
if [ -f "$file_path" ]; then
        file_contents=$(cat "$file_path")
        echo "File contents read into variable."
else
        echo "File does not exist."
fi

In [ ]:
%%bash

#cd $CEPH_JTEST_ROOT

bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_00_b4params.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_01_b4params.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_02_b4params.json


# set the scheduling parameters

bin/ceph osd pool set pl1 scrub_min_interval 1
bin/ceph osd pool set pl1 scrub_max_interval 2
bin/ceph osd pool set pl1 deep_scrub_interval 3

#bin/ceph osd pool set pl2 scrub_min_interval 2
#bin/ceph osd pool set pl2 scrub_max_interval 4
#bin/ceph osd pool set pl2 deep_scrub_interval 10

sleep 3

bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_00.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_01.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_02.json





In [ ]:
%%bash

#%cd $CEPH_JTEST_ROOT

# common scrub configs
bin/ceph tell osd.* config set osd_blocked_scrub_grace_period 20
bin/ceph tell osd.* config set osd_stats_update_period_scrubbing 2
bin/ceph tell osd.* config set osd_stats_update_period_not_scrubbing 3
bin/ceph tell osd.* config set osd_scrub_backoff_ratio 0.9999
#bin/ceph tell osd.* config set osd_scrub_interval_randomize_ratio 0.1
#bin/ceph tell osd.* config set osd_deep_scrub_randomize_ratio 0

#bin/ceph tell mgr.$(bin/ceph mgr services | jq -r .mgr) config set mgr_stats_period 2


In [ ]:
%%bash

# list the scrub queue
scrtch=to_"`date +'%H%M%S'`"
echo $scrtch
bin/ceph tell osd.0 dump_scrubs --format=json-pretty
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
bin/ceph tell osd.2 dump_scrubs --format=json-pretty
bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_0$scrtch.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_1$scrtch.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_2$scrtch.json


In [ ]:
%%bash
# set scrub parameters to guarantee slow scrub
bin/ceph tell osd.* config set osd_scrub_sleep "3.0"
bin/ceph tell osd.* config set osd_max_scrubs 1
bin/ceph tell osd.* config set osd_scrub_chunk_max 5
bin/ceph tell osd.* config set osd_shallow_scrub_chunk_max 5


In [ ]:
%%bash

pl1_num=`bin/ceph osd pool stats pl1 | sed -n -r -e 's/.*id[^0-9]*([0-9]+)$/\1/p'`
echo $pl1_num
bin/ceph tell osd.* config set osd_scrub_sleep "3.0"
bin/ceph tell osd.* config set osd_max_scrubs 1
bin/ceph tell osd.* config set osd_scrub_chunk_max 5
bin/ceph tell osd.* config set osd_shallow_scrub_chunk_max 5

bin/ceph tell osd.* config set osd_stats_update_period_scrubbing 2
bin/ceph tell osd.* config set osd_stats_update_period_not_scrubbing 2
#bin/ceph tell mgr.$(bin/ceph mgr services | jq -r .mgr) config set mgr_stats_period 2
sleep 1

# set higher urgency to one of the PGs
bin/ceph tell $pl1_num.7 scrub
bin/ceph tell $pl1_num.3 deep-scrub
bin/ceph tell $pl1_num.1 deep-scrub
bin/ceph tell $pl1_num.6 schedule-deep-scrub
echo '---------------'
bin/ceph tell osd.0 dump_scrubs --format=json-pretty
echo '---------------'
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
echo '---------------'
bin/ceph tell osd.2 dump_scrubs --format=json-pretty
sleep 1
bin/ceph pg dump pgs
echo '---------------'
#bin/ceph tell osd.0 dump_scrubs --format=json-pretty
echo '---------------'
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
echo '---------------'
#bin/ceph tell osd.2 dump_scrubs --format=json-pretty
sleep 2
bin/ceph pg dump pgs
echo '---------------'
bin/ceph tell osd.0 dump_scrubs --format=json-pretty
echo '---------------'
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
echo '---------------'
bin/ceph tell osd.2 dump_scrubs --format=json-pretty
sleep 2
bin/ceph pg dump pgs
echo '---------------'
#bin/ceph tell osd.0 dump_scrubs --format=json-pretty
echo '---------------'
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
echo '---------------'
#bin/ceph tell osd.2 dump_scrubs --format=json-pretty



In [ ]:
%%bash

scrtch=to_"`date +'%H%M%S'`"
echo $scrtch

# list the scrub queue
bin/ceph tell osd.0 dump_scrubs --format=json-pretty > /tmp/ds_0$scrtch.json
bin/ceph tell osd.1 dump_scrubs --format=json-pretty > /tmp/ds_1$scrtch.json
bin/ceph tell osd.2 dump_scrubs --format=json-pretty > /tmp/ds_2$scrtch.json
bin/ceph tell osd.0 dump_scrubs --format=json-pretty
bin/ceph tell osd.1 dump_scrubs --format=json-pretty
bin/ceph tell osd.2 dump_scrubs --format=json-pretty


In [ ]:
raise SystemExit("Stop here")

In [ ]:
%%bash

#cat /tmp/ds_01.json | jq -s '[.[]|.[]|select(.eligible==true)]| sort_by(.overdue,.pgid)' |  head -20

#cat /tmp/ds_01.json | jq -s '[.[]|.[]|select(.eligible==true)|  { pgid, overdue, sched_time } ]' |  head -20

#cat /tmp/ds_01.json | jq -s '[.[]|.[]|select(.eligible==true)|  { pgid, overdue, sched_time } ]' |  head -20

#cat ds_01.json | jq -s '[.[]|.[]|select(.eligible==false) | .overdue as $ov | . += { "ov":$ov } | [.] ] '|  head -20

#cat ds_01.json | jq -s '[.[]|.[]|select(.eligible==false) | .overdue as $ov | . += { "ov":$ov|not } | [.] ] '|  head -20

cat /tmp/ds_01.json | jq -s '[.[]|.[]|select(.eligible==true) | .overdue as $ov | . += { "ov":$ov|not } ]| sort_by(.ov,.sched_time,.pgid) | [.]  '|  head -20

#cat /tmp/ds_01.json | jq -s '[.[]|.[]|select(.eligible==true) | .overdue as $ov | .level as $lvl | . += { "ov":$ov|not, "lvl":$lvl } ]| sort_by(.ov,.sched_time,.pgid,.lvl) | [.]  '|  head -20

cat /tmp/ds_01.json | jq -s '[.[]|.[]|select(.eligible==true) | .overdue as $ov | .level as $lvl |
 . += { "ov":$ov|not, "lvl":$lvl } ]| sort_by(.ov,.sched_time,.pgid,.lvl) | [.]  '|  head -20

# use 'eligible' as just one more sort criteria
echo "========================== --- "
cat /tmp/ds_01.json | jq -s '[.[]|.[]| .eligible as $ripe | .overdue as $ov | .level as $lvl |
 . += { "not_ripe":$ripe|not, "not_ov":$ov|not, "lvl":$lvl } ] | 
 sort_by(.not_ripe, .not_ov,.sched_time,.pgid,.lvl) | [.]  '|  head -100

# now - make the sort an irregular one: if comparing the the targets of one PG, level tramps sched time



# Termination


In [ ]:
%%bash
echo '-------->' $CEPH_JTEST_ROOT
cd $CEPH_JTEST_ROOT
if [ -z $CEPH_JTEST_ROOT ]; then
    echo "CEPH_JTEST_ROOT is not set"
    [[ -f /tmp/jpath ]] && jp=`cat /tmp/jpath` || jp='.'
    echo '-------->' $jp
    cd $jp
    %env CEPH_JTEST_ROOT=$jp
fi

pwd

../src/stop.sh
sleep 4
../src/stop.sh
